# Agent Fine-Tuning with xaytune

Agent fine-tuning teaches language models to use tools and follow multi-step reasoning patterns. xaytune supports three agent data formats:

- **function_calling**: OpenAI-style tool use with `tool_calls` and `tool` messages
- **react**: Thought/Action/Observation loops for reasoning traces
- **trajectory**: Multi-turn tool use sessions for complex workflows

All three formats use **loss masking** to train only on the assistant's actions (thoughts, tool calls, final responses), not on user prompts or tool results. This focuses the model on learning decision-making and tool use patterns.

In [ ]:
from xaytune.data.agent_formats import format_function_calling

sample = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant with tools."},
        {"role": "user", "content": "What's the weather in London?"},
        {"role": "assistant", "content": None, "tool_calls": [
            {"id": "call_1", "type": "function", "function": {
                "name": "get_weather", "arguments": "{\"city\": \"London\"}"
            }}
        ]},
        {"role": "tool", "tool_call_id": "call_1", "content": "{\"temp\": 18, \"condition\": \"cloudy\"}"},
        {"role": "assistant", "content": "It's 18 degrees and cloudy in London."},
    ]
}

messages = format_function_calling(sample)
print("Function Calling Format:")
for msg in messages:
    marker = "TRAIN" if msg.trainable else "MASK "
    preview = msg.content[:70].replace("\n", " ")
    print(f"  [{marker}] {msg.role}: {preview}...")

## ReAct Format

ReAct (Reasoning + Acting) traces capture the model's internal reasoning process through explicit Thought/Action/Observation loops. Each step includes:

- **Thought**: The model's reasoning about what to do next
- **Action**: The tool or action to execute
- **Action Input**: Arguments for the action
- **Observation**: The result returned by the tool

Only the Thought, Action, and Action Input are trainable — Observations are masked since they come from the environment.

In [ ]:
from xaytune.data.agent_formats import format_react

sample = {
    "task": "Find the population of France",
    "steps": [
        {
            "thought": "I need to search for France's population.",
            "action": "search",
            "action_input": "France population 2024",
            "observation": "France has approximately 68 million people."
        },
        {
            "thought": "I now have the answer.",
            "action": "finish",
            "action_input": "France has approximately 68 million people."
        }
    ]
}

messages = format_react(sample)
print("ReAct Format:")
for msg in messages:
    marker = "TRAIN" if msg.trainable else "MASK "
    preview = msg.content[:70].replace("\n", " ")
    print(f"  [{marker}] {msg.role}: {preview}...")

## Trajectory Format

Trajectories capture multi-step tool use sessions where the assistant performs a sequence of actions to accomplish a goal. This format is ideal for:

- Code generation workflows with file creation and testing
- Multi-step data analysis tasks
- Complex debugging sessions
- Any task requiring multiple tool invocations

The trajectory format masks user input and tool results, training only on the assistant's decisions and responses.

In [ ]:
from xaytune.data.agent_formats import format_trajectory

sample = {
    "system": "You are a coding assistant with terminal access.",
    "goal": "Create a Python file that prints hello world",
    "turns": [
        {"role": "assistant", "content": "I'll create the file.", "tool_calls": [
            {"name": "write_file", "arguments": {"path": "hello.py", "content": "print('hello world')"}}
        ]},
        {"role": "tool", "content": "File written successfully."},
        {"role": "assistant", "content": "Let me run it to verify.", "tool_calls": [
            {"name": "run_command", "arguments": {"cmd": "python hello.py"}}
        ]},
        {"role": "tool", "content": "hello world"},
        {"role": "assistant", "content": "Done! The file prints 'hello world' as expected."},
    ]
}

messages = format_trajectory(sample)
print("Trajectory Format:")
for msg in messages:
    marker = "TRAIN" if msg.trainable else "MASK "
    preview = msg.content[:70].replace("\n", " ")
    print(f"  [{marker}] {msg.role}: {preview}...")

## Loss Masking

xaytune's agent tokenizer automatically masks tokens that shouldn't contribute to training loss. Non-trainable tokens (user prompts, tool results) are set to `IGNORE_INDEX=-100`, which causes them to be ignored during loss computation.

This ensures the model learns to:
- Generate appropriate reasoning (Thoughts)
- Select correct tools and actions
- Format tool arguments properly
- Synthesize final responses from tool results

...without learning to predict user input or tool outputs.

In [ ]:
from unittest.mock import MagicMock
from xaytune.data.agent_tokenizer import tokenize_agent_dataset

tok = MagicMock()
tok.model_max_length = 512

def tokenize(text, **kwargs):
    words = text.split()
    return {"input_ids": list(range(len(words)))}

tok.side_effect = tokenize

messages = format_function_calling({
    "messages": [
        {"role": "user", "content": "What is the weather in London today?"},
        {"role": "assistant", "content": None, "tool_calls": [
            {"id": "c1", "type": "function", "function": {
                "name": "get_weather",
                "arguments": "{\"city\": \"London\"}"
            }}
        ]},
        {"role": "tool", "tool_call_id": "c1", "content": "{\"temp\": 18}"},
        {"role": "assistant", "content": "It is 18 degrees in London."},
    ]
})

result = tokenize_agent_dataset([messages], tok)
sample = result[0]
total = len(sample["input_ids"])
trainable = sum(1 for v in sample["labels"] if v != -100)
masked = sum(1 for v in sample["labels"] if v == -100)
print(f"Total tokens: {total}")
print(f"Trainable tokens: {trainable} ({100*trainable//total}%)")
print(f"Masked tokens: {masked} ({100*masked//total}%)")
print(f"\nThe model learns to predict ONLY the assistant's actions,")
print(f"not the user prompts or tool results.")

In [ ]:
# Real-world agent fine-tuning:
#
# import xaytune
#
# state = xaytune.finetune(
#     model="meta-llama/Llama-3.1-8B",
#     dataset="data/agent_traces.jsonl",
#     method="lora",
#     format="function_calling",  # or "react" or "trajectory"
#     num_epochs=3,
# )
print("To fine-tune on agent data, just set format='function_calling'")
print("(or 'react' or 'trajectory') in xaytune.finetune().")
print("Loss masking is applied automatically.")

## CLI Configuration

You can also configure agent fine-tuning via YAML:

```yaml
recipe: finetune
method: lora
model:
  name: meta-llama/Llama-3.1-8B
data:
  path: data/agent_traces.jsonl
  format: function_calling
training:
  num_epochs: 3
  learning_rate: 2e-4
```

Then run:

```bash
xaytune train --config config.yaml
```

## Next Steps

- **02_finetuning.ipynb**: Learn the basics of fine-tuning with xaytune
- **09_callbacks.ipynb**: Monitor training with custom callbacks
- **xaytune documentation**: Full API reference at https://github.com/szaher/xaytune

For production agent fine-tuning, consider:
- Collecting diverse tool use examples
- Balancing successful and failed trajectories
- Validating agent behavior on held-out tasks
- Monitoring tool call accuracy and reasoning quality